# Structure Type Study - Automated Checklist

**ODOT BDM Section 201 — Structure Type Study Review Checklist (November 1, 2022)**

This notebook automates the data-gathering steps of the ODOT Structure Type Study checklist.
Enter a bridge SFN (or location coordinates) in the cell below, and each section will
query TIMS, NHD, USGS, and other ODOT systems to pre-populate the required information.

Items that cannot be fully automated are marked with `# TODO` and manual-entry fields.

---
## 0. Setup & Configuration

Enter your project parameters here. All downstream cells reference these values.

In [1]:
# ============================================================
# USER INPUT — Fill in these fields for your project
# ============================================================

# Existing bridge SFN (Structure File Number) — leave blank for new locations
EXISTING_SFN = "2102226"  # e.g. "2102226"

# If no existing SFN, provide a lat/lon for the project site
PROJECT_LAT = None   # e.g. 40.182
PROJECT_LON = None   # e.g. -82.949

# Project identification
PID = ""             # e.g. "110000"
CONSULTANT = ""
REVIEWER = ""

# Design parameters
DESIGN_LIVE_LOAD = "HL-93"  # Vehicular live load designation
FUTURE_WEARING_SURFACE_PSF = 60  # psf, per BDM
DESIGN_SPEED_MPH = 55

# Search radii for nearby features (miles)
NEARBY_BRIDGE_RADIUS = 5.0
STREAM_SEARCH_RADIUS = 0.5
PROJECT_SEARCH_RADIUS = 5.0

In [2]:
# ============================================================
# Imports and initialization
# ============================================================
import sys
import json
from datetime import datetime, timedelta
from IPython.display import display, Markdown, HTML

# Import ODOT objects
from ODOT import (
    TIMSBridge, TIMSProject, TIMSRoad, TIMSWaterway,
    NHDFlowline, USGSStreamStats, USGSGauge,
    BridgeEngineeringChecks, ArcGISLayer, TIMS_URLS,
)

# Helper to render checklist items
def checklist(description, value=None, status="auto", bdm_ref=""):
    """Render a single checklist item as formatted markdown.
    
    status: 'auto' (data retrieved), 'manual' (needs user input), 
            'todo' (not yet implemented), 'na' (not applicable)
    """
    icons = {"auto": "\u2705", "manual": "\u270f\ufe0f", "todo": "\u26a0\ufe0f", "na": "\u2796"}
    icon = icons.get(status, "\u2610")
    ref = f" *({bdm_ref})*" if bdm_ref else ""
    val = f": **{value}**" if value is not None else ""
    display(Markdown(f"{icon} {description}{val}{ref}"))

def section_header(title):
    display(Markdown(f"### {title}"))
    display(Markdown("---"))

print("Imports loaded successfully.")

Imports loaded successfully.


In [3]:
# ============================================================
# Fetch existing bridge data from TIMS (if SFN provided)
# ============================================================
bridge = None
lat, lon = PROJECT_LAT, PROJECT_LON

if EXISTING_SFN:
    try:
        bridge = TIMSBridge(EXISTING_SFN)
        lat = bridge.lat
        lon = bridge.lon
        print(f"Loaded existing bridge: {bridge}")
    except ValueError as e:
        print(f"WARNING: Could not load SFN {EXISTING_SFN}: {e}")
        print("Falling back to manual coordinates.")
elif lat and lon:
    print(f"No existing SFN provided. Using coordinates: ({lat}, {lon})")
else:
    print("WARNING: No SFN or coordinates provided. Many sections will be incomplete.")
    print("Please set EXISTING_SFN or PROJECT_LAT/PROJECT_LON above.")

Loaded existing bridge: TIMSBridge(sfn='2102226', 'I-71 NB' over '3.55 MI N OF FRANKLIN CO', built=1959, D/S/S=8/8/8, suff=89.9)


---
## 1. General (BDM 201.1.1)

Each structure requiring a separate SFN per the Ohio Bridge Coding Guide gets its own
Structure Type Study. Identical parallel bridges needing separate SFNs: develop a full
STS for one, and reference it from the twin.

In [4]:
section_header("1. General — Bridge Identification (201.1.1)")

if bridge:
    checklist("Structure File Number (SFN)", EXISTING_SFN, "auto", "201.1.1")
    checklist("Facility Carried", bridge.facility_carried, "auto")
    checklist("Feature Intersected", bridge.feature_intersected, "auto")
    checklist("County", bridge.county, "auto")
    checklist("District", bridge.district, "auto")
    checklist("Year Built", bridge.year_built, "auto")
    checklist("Owner", bridge.owner, "auto")
    checklist("Maintenance Responsibility", bridge.maintenance_responsibility, "auto")
    checklist("Latitude", bridge.lat, "auto")
    checklist("Longitude", bridge.lon, "auto")
    checklist("Overall Length (ft)", bridge.overall_length, "auto")
    checklist("Max Span Length (ft)", bridge.max_span_length, "auto")
    checklist("Total Spans", bridge.total_spans, "auto")
    checklist("Deck Width (ft)", bridge.deck_width, "auto")
    checklist("Roadway Width (ft)", bridge.roadway_width, "auto")
    checklist("Deck Area (sq ft)", bridge.deck_area, "auto")
    checklist("Skew (degrees)", bridge.skew, "auto")
    checklist("Lanes On", bridge.lanes_on, "auto")
    checklist("Lanes Under", bridge.lanes_under, "auto")
    checklist("Main Material Code", bridge.main_material, "auto")
    checklist("Main Type Code", bridge.main_type, "auto")
    checklist("Design Load", bridge.design_load, "auto")
    checklist("Functional Class", bridge.functional_class, "auto")
    checklist("NHS Designation", bridge.nhs_designation, "auto")
else:
    checklist("No existing bridge loaded — enter data manually", status="manual")

# PID / Project info
checklist("PID", PID or "(not set)", "manual")
checklist("Consultant", CONSULTANT or "(not set)", "manual")
checklist("Reviewer", REVIEWER or "(not set)", "manual")
checklist("Separate STS for each SFN (identical twins reference one full STS)",
          status="manual", bdm_ref="201.1.1")

### 1. General — Bridge Identification (201.1.1)

---

✅ Structure File Number (SFN): **2102226** *(201.1.1)*

✅ Facility Carried: **I-71 NB**

✅ Feature Intersected: **3.55 MI N OF FRANKLIN CO**

...

✅ NHS Designation: **1**

✏️ PID: **(not set)**

✏️ Consultant: **(not set)**

✏️ Reviewer: **(not set)**

✏️ Separate STS for each SFN (identical twins reference one full STS) *(201.1.1)*

### Existing Bridge Condition Summary

In [5]:
if bridge:
    section_header("Existing Bridge Condition (from TIMS)")
    ratings = bridge.condition_ratings
    for key, val in ratings.items():
        checklist(f"{key.title()} Rating", val, "auto")
    checklist("Sufficiency Rating", bridge.sufficiency_rating, "auto")
    checklist("Structurally Deficient", bridge.is_structurally_deficient, "auto")
    checklist("Scour Critical", bridge.is_scour_critical, "auto")
    checklist("Fracture Critical", bridge.fracture_critical, "auto")
    checklist("Underwater Inspection Required", bridge.underwater_inspection, "auto")
    
    # Load rating
    checklist("Inventory Rating Factor", bridge.inventory_rating_factor, "auto")
    checklist("Operating Rating Factor", bridge.operating_rating_factor, "auto")
    
    # Traffic
    checklist("ADT", bridge.adt, "auto")
    checklist("Future ADT", bridge.future_adt, "auto")
    checklist("Approach Roadway Width (ft)", bridge.approach_roadway_width, "auto")
    checklist("Bypass/Detour Length", bridge.bypass_detour_length, "auto")
else:
    print("No existing bridge data available.")

### Existing Bridge Condition (from TIMS)

---

✅ Deck Rating: **8**

✅ Superstructure Rating: **8**

✅ Substructure Rating: **8**

...

✅ Operating Rating Factor: **2320.0**

✅ ADT: **41597**

✅ Future ADT: **57737**

✅ Approach Roadway Width (ft): **50.0**

✅ Bypass/Detour Length: **01**

---
## 2. Structure Type Study Submission Requirements (BDM 201.1.2)

The following items are required for each STS submittal. This section tracks
which deliverables have been prepared.

In [6]:
section_header("2. Submission Requirements Checklist (201.1.2)")

checklist("Profile for each bridge alternative", status="manual", bdm_ref="201.1.2.A / 201.1.2.1")
checklist("Preliminary Structure Site Plan for preferred alternative", status="manual", bdm_ref="201.1.2.B / 201.1.2.2")
checklist("Hydrology & Hydraulic Report", status="manual", bdm_ref="201.1.2.C / L&D Vol. 2")
checklist("Narrative of Bridge Alternatives", status="manual", bdm_ref="201.1.2.D / 201.1.2.3")
checklist("Cost Analysis", status="manual", bdm_ref="201.1.2.E / 201.1.2.4")
checklist("Foundation Recommendations", status="manual", bdm_ref="201.1.2.F / 201.1.2.5")
checklist("Preliminary Maintenance of Traffic Plan", status="manual", bdm_ref="201.1.2.G / 201.1.2.6")

display(Markdown("\n**Concurrent submissions to Office of Structural Engineering:**"))
checklist("Non-standard railing types", status="manual", bdm_ref="309.4.1")

display(Markdown("\n**May be required:**"))
checklist("Retaining Wall Justification", status="manual", bdm_ref="L&D Vol 3 1407.2")
checklist("Noise Barrier Justification", status="manual", bdm_ref="ODOT Noise Analysis Manual")
checklist("Pedestrian Overpass Justification", status="manual", bdm_ref="L&D Vol 3 1407.4")

### 2. Submission Requirements Checklist (201.1.2)

---

✏️ Profile for each bridge alternative *(201.1.2.A / 201.1.2.1)*

✏️ Preliminary Structure Site Plan for preferred alternative *(201.1.2.B / 201.1.2.2)*

✏️ Hydrology & Hydraulic Report *(201.1.2.C / L&D Vol. 2)*

...

✏️ Non-standard railing types *(309.4.1)*


**May be required:**

✏️ Retaining Wall Justification *(L&D Vol 3 1407.2)*

✏️ Noise Barrier Justification *(ODOT Noise Analysis Manual)*

✏️ Pedestrian Overpass Justification *(L&D Vol 3 1407.4)*

---
## 3. Profile for Each Bridge Alternative (BDM 201.1.2.1)

This section gathers waterway, clearance, and span data needed for bridge profiles.
Hydraulic data is pulled from USGS StreamStats and NHD where available.

In [7]:
section_header("3a. Profile General Requirements (201.1.2.1)")

checklist("Profile drawn to same scale as corresponding Plan view", status="manual", bdm_ref="201.1.2.1")
checklist("Existing and proposed profile grade lines", status="manual", bdm_ref="201.1.2.1.A")
checklist("Existing ground line", status="manual", bdm_ref="201.1.2.1.B")
checklist("Outline of the structure", status="manual", bdm_ref="201.1.2.1.C")
checklist("Existing and proposed profile grade elevations at 25-ft increments", status="manual", bdm_ref="201.1.2.1.F")
checklist("Minimum and required vertical/lateral clearances", status="manual", bdm_ref="201.1.2.1.G")

### 3a. Profile General Requirements (201.1.2.1)

---

✏️ Profile drawn to same scale as corresponding Plan view *(201.1.2.1)*

✏️ Existing and proposed profile grade lines *(201.1.2.1.A)*

✏️ Existing ground line *(201.1.2.1.B)*

✏️ Outline of the structure *(201.1.2.1.C)*

✏️ Existing and proposed profile grade elevations at 25-ft increments *(201.1.2.1.F)*

✏️ Minimum and required vertical/lateral clearances *(201.1.2.1.G)*

In [8]:
# Span measurements
section_header("3b. Span Dimensions (201.1.2.1.H)")

if bridge:
    checklist("Existing span count", bridge.total_spans, "auto")
    checklist("Existing max span length (ft)", bridge.max_span_length, "auto")
    checklist("Existing overall length (ft)", bridge.overall_length, "auto")
    checklist("Existing skew (degrees)", bridge.skew, "auto")

checklist("Spans on tangent: c/c bearings along CL construction", status="manual", bdm_ref="201.1.2.1.H.1")
checklist("Spans on curve: measure along Reference Line", status="manual", bdm_ref="201.1.2.1.H.2")
checklist("Standard concrete slab spans: 1-ft behind face of abutment", status="manual", bdm_ref="201.1.2.1.H.3")

### 3b. Span Dimensions (201.1.2.1.H)

---

✅ Existing span count

✅ Existing max span length (ft): **58.0**

✅ Existing overall length (ft): **132.0**

✅ Existing skew (degrees): **33**

✏️ Spans on tangent: c/c bearings along CL construction *(201.1.2.1.H.1)*

✏️ Spans on curve: measure along Reference Line *(201.1.2.1.H.2)*

✏️ Standard concrete slab spans: 1-ft behind face of abutment *(201.1.2.1.H.3)*

### 3c. Waterway Data — Riverine Streams (201.1.2.1.D)

Automated queries to NHD, USGS StreamStats, and TIMS for hydraulic information.

In [9]:
section_header("3c. Bridges Conveying Riverine Streams (201.1.2.1.D)")

nhd_data = []
streamstats_data = {}
usgs_gauges = []

if lat and lon:
    # NHD Flowlines near the bridge
    print("Querying NHD flowlines...")
    nhd_data = NHDFlowline.near(lon, lat, radius_miles=STREAM_SEARCH_RADIUS)
    if nhd_data:
        print(f"Found {len(nhd_data)} NHD flowlines within {STREAM_SEARCH_RADIUS} mi:")
        for s in nhd_data:
            name = s.get('GNIS_Name') or '(unnamed)'
            order = s.get('StreamOrder', '?')
            print(f"  - {name} (Stream Order: {order})")
    else:
        print("No NHD flowlines found nearby.")
    
    # NHD Waterbodies
    print("\nQuerying NHD waterbodies...")
    waterbodies = NHDFlowline.waterbodies_near(lon, lat, radius_miles=1.0)
    if waterbodies:
        print(f"Found {len(waterbodies)} waterbodies within 1 mi:")
        for w in waterbodies:
            print(f"  - {w.get('GNIS_Name') or '(unnamed)'} ({w.get('FType')})")

    # USGS StreamStats watershed delineation
    print("\nQuerying USGS StreamStats (may take 1-2 minutes)...")
    streamstats_data = USGSStreamStats.delineate(lon, lat)
    if streamstats_data.get("drainage_area_sqmi"):
        print(f"Drainage Area: {streamstats_data['drainage_area_sqmi']} sq mi")
    else:
        print("StreamStats delineation did not return drainage area.")
    
    # USGS Stream Gauges
    print("\nQuerying nearby USGS stream gauges...")
    usgs_gauges = USGSGauge.near(lon, lat, radius_miles=10)
    if usgs_gauges:
        print(f"Found {len(usgs_gauges)} active gauges within 10 mi")
else:
    print("No coordinates available — skipping waterway queries.")

### 3c. Bridges Conveying Riverine Streams (201.1.2.1.D)

---

Querying NHD flowlines...
No NHD flowlines found nearby.

Querying NHD waterbodies...
Found 17 waterbodies within 1 mi:
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)
  - (unnamed) (None)

Querying USGS StreamStats (may take 1-2 minutes)...
StreamStats delineation did not return drainage area.

Querying nearby USGS stream gauges...


USGS gauge query failed: 400 Client Error:  for url: https://waterservices.usgs.gov/nwis/site/?format=json&bBox=-83.13240318315019%2C40.03682846376812%2C-82.76610281684981%2C40.32668353623188&siteType=ST&siteStatus=active&hasDataTypeCd=iv&parameterCd=00060&siteOutput=expanded


In [10]:
# Waterway checklist items (201.1.2.1.D)
section_header("Waterway Profile Items (201.1.2.1.D)")

# TIMS-sourced waterway data
if bridge:
    checklist("Drainage Area (TIMS)", f"{bridge.drainage_area} sq mi" if bridge.drainage_area else None, 
              "auto" if bridge.drainage_area else "manual")
    checklist("Stream Velocity (TIMS)", f"{bridge.stream_velocity} fps" if bridge.stream_velocity else None,
              "auto" if bridge.stream_velocity else "manual")
    checklist("Waterway Adequacy", bridge.waterway_adequacy, "auto")
    checklist("Channel Rating", bridge.channel_rating, "auto")
    checklist("Scour Critical Code", bridge.scour_critical, "auto")

# StreamStats-sourced data
da = streamstats_data.get("drainage_area_sqmi")
if da:
    checklist("Drainage Area (USGS StreamStats)", f"{da} sq mi", "auto")

# Items requiring manual input or H&H analysis
checklist("Cross-section of waterway channel", status="manual", bdm_ref="201.1.2.1.D.1")
checklist("Highest known high-water mark", status="manual", bdm_ref="201.1.2.1.D.2")
checklist("Normal water elevation", status="manual", bdm_ref="201.1.2.1.D.3 / 101.7")
checklist("Ordinary high-water mark (OHWM)", status="manual", bdm_ref="201.1.2.1.D.4 / 101.7")
checklist("Thalweg elevation", status="manual", bdm_ref="201.1.2.1.D.5 / 101.7")
checklist("Scour information", status="manual", bdm_ref="201.1.2.1.D.6")
checklist("Design and 100-year highwater elevations (incl. backwater)", status="manual", bdm_ref="201.1.2.1.D.7")
checklist("Overtopping flood elevation and frequency", status="manual", bdm_ref="201.1.2.1.D.8")

### Waterway Profile Items (201.1.2.1.D)

---

✏️ Drainage Area (TIMS)

✏️ Stream Velocity (TIMS)

✅ Waterway Adequacy: **N**

...

✏️ Ordinary high-water mark (OHWM) *(201.1.2.1.D.4 / 101.7)*

✏️ Thalweg elevation *(201.1.2.1.D.5 / 101.7)*

✏️ Scour information *(201.1.2.1.D.6)*

✏️ Design and 100-year highwater elevations (incl. backwater) *(201.1.2.1.D.7)*

✏️ Overtopping flood elevation and frequency *(201.1.2.1.D.8)*

In [11]:
# Peak flow statistics from StreamStats (if available)
section_header("Peak Flow Statistics (from USGS StreamStats)")

peak_flows = streamstats_data.get("peak_flows", {})
if peak_flows:
    for name, value in sorted(peak_flows.items()):
        checklist(name, f"{value:.0f} cfs", "auto")
else:
    print("No peak flow statistics available from StreamStats.")
    print("Peak flows must be determined from H&H analysis.")

# Basin characteristics
basin = streamstats_data.get("basin_characteristics", {})
if basin:
    display(Markdown("**Basin Characteristics:**"))
    for code, value in sorted(basin.items()):
        if value is not None:
            print(f"  {code}: {value}")

### Peak Flow Statistics (from USGS StreamStats)

---

No peak flow statistics available from StreamStats.
Peak flows must be determined from H&H analysis.


In [12]:
# Controlled outlets / spillways (201.1.2.1.E)
section_header("3d. Controlled Outlets / Spillways (201.1.2.1.E)")

checklist("Cross-section of the channel", status="manual", bdm_ref="201.1.2.1.E.1")
checklist("Summer pool elevation", status="manual", bdm_ref="201.1.2.1.E.2")
checklist("Winter pool elevation", status="manual", bdm_ref="201.1.2.1.E.3")
checklist("Surveillance pool elevation", status="manual", bdm_ref="201.1.2.1.E.4")
checklist("Record pool elevation", status="manual", bdm_ref="201.1.2.1.E.5")
checklist("Spillway elevation", status="manual", bdm_ref="201.1.2.1.E.6")
checklist("Top of dam elevation", status="manual", bdm_ref="201.1.2.1.E.7")

### 3d. Controlled Outlets / Spillways (201.1.2.1.E)

---

✏️ Cross-section of the channel *(201.1.2.1.E.1)*

✏️ Summer pool elevation *(201.1.2.1.E.2)*

✏️ Winter pool elevation *(201.1.2.1.E.3)*

✏️ Surveillance pool elevation *(201.1.2.1.E.4)*

✏️ Record pool elevation *(201.1.2.1.E.5)*

✏️ Spillway elevation *(201.1.2.1.E.6)*

✏️ Top of dam elevation *(201.1.2.1.E.7)*

### 3e. Environmental Constraints

Scenic rivers, mussel streams, wetlands, and HUC watershed info from TIMS and NHD.

In [13]:
section_header("Environmental Features Near Project Site")

if lat and lon:
    # Scenic Rivers
    scenic = TIMSWaterway.scenic_rivers_near(lon, lat, radius_miles=2.0)
    if scenic:
        checklist(f"Scenic Rivers within 2 mi: {len(scenic)} found", status="auto")
        for s in scenic:
            print(f"  - {s.get('RIVER_NAME', s.get('NAME', 'Unknown'))}")
    else:
        checklist("No scenic rivers within 2 miles", status="auto")
    
    # Mussel Streams
    mussel = TIMSWaterway.mussel_streams_near(lon, lat, radius_miles=2.0)
    if mussel:
        checklist(f"Mussel habitat streams within 2 mi: {len(mussel)} found", status="auto")
    else:
        checklist("No mussel streams within 2 miles", status="auto")
    
    # Wetlands
    wetlands = TIMSWaterway.wetlands_near(lon, lat, radius_miles=1.0)
    if wetlands:
        checklist(f"NWI Wetlands within 1 mi: {len(wetlands)} found", status="auto")
    else:
        checklist("No NWI wetlands within 1 mile", status="auto")
    
    # HUC-12 Watershed
    huc12 = NHDFlowline.huc12_at(lon, lat)
    if huc12:
        for h in huc12:
            checklist("HUC-12 Watershed", h.get('Name', h.get('HUC12', 'Unknown')), "auto")
else:
    print("No coordinates available.")

### Environmental Features Near Project Site

---

✅ No scenic rivers within 2 miles

✅ Mussel habitat streams within 2 mi: 1 found

✅ NWI Wetlands within 1 mi: 41 found

✅ HUC-12 Watershed: **Unknown**

---
## 4. Site Plan — Preferred Bridge Alternative (BDM 201.1.2.2)

This section gathers data for the structure site plan. Drawing/CAD work is manual,
but many data fields can be pre-populated.

In [14]:
section_header("4a. Site Plan General (201.1.2.2)")

checklist("1 in = 20 ft Site Plan scale", status="manual", bdm_ref="201.1.2.2")
checklist("1 in = 10 ft scale (if 1=20 too small)", status="manual", bdm_ref="C201.1.2.2")
checklist("Match-marked across sheets for bridges > 400 ft", status="manual", bdm_ref="201.1.2.2")

display(Markdown("\n**Plan View Items (201.1.2.2.A):**"))
checklist("Plan view with existing structure (dashed)", status="manual", bdm_ref="201.1.2.2.A.1")
checklist("Contours showing existing ground (2-ft intervals; 5-ft for steep slopes)", status="manual", bdm_ref="201.1.2.2.A.2")
checklist("Existing utility lines and their disposition", status="manual", bdm_ref="201.1.2.2.A.3 / 310.4")
checklist("Proposed structure", status="manual", bdm_ref="201.1.2.2.A.4")
checklist("Proposed temporary bridge", status="manual", bdm_ref="201.1.2.2.A.5")
checklist("Proposed channel improvements", status="manual", bdm_ref="201.1.2.2.A.6")
checklist("North arrow", status="manual", bdm_ref="201.1.2.2.A.7")
checklist("Location and value of required/actual min lateral and vertical clearances", status="manual", bdm_ref="201.1.2.2.A.8")
checklist("All other pertinent features", status="manual", bdm_ref="201.1.2.2.A.9")

### 4a. Site Plan General (201.1.2.2)

---

✏️ 1 in = 20 ft Site Plan scale *(201.1.2.2)*

✏️ 1 in = 10 ft scale (if 1=20 too small) *(C201.1.2.2)*

✏️ Match-marked across sheets for bridges > 400 ft *(201.1.2.2)*

...

✏️ Proposed temporary bridge *(201.1.2.2.A.5)*

✏️ Proposed channel improvements *(201.1.2.2.A.6)*

✏️ North arrow *(201.1.2.2.A.7)*

✏️ Location and value of required/actual min lateral and vertical clearances *(201.1.2.2.A.8)*

✏️ All other pertinent features *(201.1.2.2.A.9)*

In [15]:
# Clearance check for railroad crossings
section_header("4b. Railroad Clearances")

if lat and lon:
    rail_layer = ArcGISLayer(TIMS_URLS["rail_crossings"])
    rail_crossings = rail_layer.query_by_point(lon, lat, buffer_miles=1.0)
    if rail_crossings:
        checklist(f"Railroad crossings within 1 mi: {len(rail_crossings)} found", status="auto")
        for rc in rail_crossings:
            xing_id = rc.get('CROSSING_ID', rc.get('DOT_CROSSING_ID', 'Unknown'))
            rr = rc.get('RAILROAD', rc.get('RR_COMPANY', 'Unknown'))
            print(f"  - Crossing {xing_id} ({rr})")
        checklist("Vertical clearance per AREMA ch. 15 sec 1.2.6(a) — measured from top of highest rail, 6 ft from CL track",
                  status="manual", bdm_ref="C201.1.2.2.A.8")
    else:
        checklist("No railroad crossings within 1 mile", status="auto")
    
    # Also check rail lines
    rail_lines_layer = ArcGISLayer(TIMS_URLS["rail_lines"])
    rail_lines = rail_lines_layer.query_by_point(lon, lat, buffer_miles=0.5)
    if rail_lines:
        checklist(f"Rail lines within 0.5 mi: {len(rail_lines)} found", status="auto")
else:
    print("No coordinates available.")

### 4b. Railroad Clearances

---

✅ No railroad crossings within 1 mile

In [16]:
section_header("4c. Site Plan Profile & Curve Data (201.1.2.2.B-C)")

checklist("Profile per BDM 201.1.2.1 with same scale as plan view", status="manual", bdm_ref="201.1.2.2.B")
checklist("Horizontal and vertical curve data", status="manual", bdm_ref="201.1.2.2.C")

### 4c. Site Plan Profile & Curve Data (201.1.2.2.B-C)

---

✏️ Profile per BDM 201.1.2.1 with same scale as plan view *(201.1.2.2.B)*

✏️ Horizontal and vertical curve data *(201.1.2.2.C)*

In [17]:
section_header("4d. Site Plan Waterway Data (201.1.2.2.D-F)")

display(Markdown("**Riverine streams (201.1.2.2.D):**"))
checklist("Elevations per BDM 201.1.2.1.D", status="manual", bdm_ref="201.1.2.2.D.1")

# Size of drainage area — auto-populated if available
da_tims = bridge.drainage_area if bridge else None
da_usgs = streamstats_data.get("drainage_area_sqmi")
da_display = da_usgs or da_tims
checklist("Size of drainage area", f"{da_display} sq mi" if da_display else None,
          "auto" if da_display else "manual", bdm_ref="201.1.2.2.D.2")

checklist("Design year flood frequency, discharge, and velocity", status="manual", bdm_ref="201.1.2.2.D.3")
checklist("100-yr discharge and velocity (label 'FIS' if from FEMA Insurance Study)", status="manual", bdm_ref="201.1.2.2.D.4")
checklist("Freeboard for design year water surface elevation", status="manual", bdm_ref="201.1.2.2.D.5")

display(Markdown("\n**Controlled outlets / spillways (201.1.2.2.E):**"))
checklist("Elevations per BDM 201.1.2.1.E", status="manual", bdm_ref="201.1.2.2.E.1")
checklist("Size of drainage area", status="manual", bdm_ref="201.1.2.2.E.2")
checklist("Water body name", status="manual", bdm_ref="201.1.2.2.E.3")
checklist("Owner of outlet or spillway", status="manual", bdm_ref="201.1.2.2.E.4")
checklist("Note indicating pool elevation controlled by outlet/spillway", status="manual", bdm_ref="201.1.2.2.E.5")

display(Markdown("\n**Buried culvert bridges (201.1.2.2.F):**"))
checklist("Service life of 75 years, water pH, abrasion level (1-6)", status="manual", bdm_ref="201.1.2.2.F")

### 4d. Site Plan Waterway Data (201.1.2.2.D-F)

---

**Riverine streams (201.1.2.2.D):**

✏️ Elevations per BDM 201.1.2.1.D *(201.1.2.2.D.1)*

✏️ Size of drainage area *(201.1.2.2.D.2)*

...

✏️ Water body name *(201.1.2.2.E.3)*

✏️ Owner of outlet or spillway *(201.1.2.2.E.4)*

✏️ Note indicating pool elevation controlled by outlet/spillway *(201.1.2.2.E.5)*


**Buried culvert bridges (201.1.2.2.F):**

✏️ Service life of 75 years, water pH, abrasion level (1-6) *(201.1.2.2.F)*

In [18]:
section_header("4e. Proposed Structure Block (201.1.2.2.G)")
display(Markdown("*Located in lower right-hand corner, 6.5 in wide on 22x34 sheet*"))

checklist("Brief description of proposed bridge type", status="manual", bdm_ref="201.1.2.2.G.1")
checklist("Span lengths and how measured (c/c bearings, f/f abutments)", status="manual", bdm_ref="201.1.2.2.G.2")
checklist("Roadway and sidewalk widths and how measured", status="manual", bdm_ref="201.1.2.2.G.3")
checklist("Vehicular Live Load designation(s)", DESIGN_LIVE_LOAD, "auto", bdm_ref="201.1.2.2.G.4")
checklist("Future Wearing Surface", f"{FUTURE_WEARING_SURFACE_PSF} psf", "auto", bdm_ref="201.1.2.2.G.5")
checklist("Skew angle and direction (right forward or left forward)", status="manual", bdm_ref="201.1.2.2.G.6")
checklist("Approach slab length, thickness, and standard drawing number", status="manual", bdm_ref="201.1.2.2.G.7")
checklist("Horizontal alignment description", status="manual", bdm_ref="201.1.2.2.G.8")
checklist("Crown or Superelevation with rate of cross-slope", status="manual", bdm_ref="201.1.2.2.G.9")

# Auto-populated from TIMS
if bridge:
    checklist("Latitude of bridge", bridge.lat, "auto", bdm_ref="201.1.2.2.G.10")
    checklist("Longitude of bridge", bridge.lon, "auto", bdm_ref="201.1.2.2.G.10")
    checklist("Deck area", f"{bridge.deck_area} sq ft" if bridge.deck_area else None,
              "auto" if bridge.deck_area else "manual", bdm_ref="201.1.2.2.G.11")
else:
    checklist("Latitude of bridge", lat, "auto" if lat else "manual", bdm_ref="201.1.2.2.G.10")
    checklist("Longitude of bridge", lon, "auto" if lon else "manual", bdm_ref="201.1.2.2.G.10")
    checklist("Deck area", status="manual", bdm_ref="201.1.2.2.G.11")

### 4e. Proposed Structure Block (201.1.2.2.G)

---

*Located in lower right-hand corner, 6.5 in wide on 22x34 sheet*

✏️ Brief description of proposed bridge type *(201.1.2.2.G.1)*

✏️ Span lengths and how measured (c/c bearings, f/f abutments) *(201.1.2.2.G.2)*

...

✏️ Horizontal alignment description *(201.1.2.2.G.8)*

✏️ Crown or Superelevation with rate of cross-slope *(201.1.2.2.G.9)*

✅ Latitude of bridge: **40.181756** *(201.1.2.2.G.10)*

✅ Longitude of bridge: **-82.949253** *(201.1.2.2.G.10)*

✅ Deck area: **8844 sq ft** *(201.1.2.2.G.11)*

In [19]:
section_header("4f. Existing Structure Block (201.1.2.2.H)")
display(Markdown("*Located above the Proposed Structure Block, 6.5 in wide on 22x34 sheet*"))

if bridge:
    checklist("Brief description of existing bridge type", 
              f"Material: {bridge.main_material}, Type: {bridge.main_type}", "auto", bdm_ref="201.1.2.2.H.1")
    checklist("Span lengths and how measured", status="manual", bdm_ref="201.1.2.2.H.2")
    checklist("Roadway width and how measured", f"{bridge.roadway_width} ft" if bridge.roadway_width else None,
              "auto" if bridge.roadway_width else "manual", bdm_ref="201.1.2.2.H.3")
    checklist("Vehicular Live Load for original design", bridge.design_load, "auto", bdm_ref="201.1.2.2.H.4")
    checklist("Skew angle and direction", f"{bridge.skew} degrees" if bridge.skew else None,
              "auto" if bridge.skew else "manual", bdm_ref="201.1.2.2.H.5")
    checklist("Type and thickness of existing wearing surface", status="manual", bdm_ref="201.1.2.2.H.6")
    checklist("Approach slab info (if known)", status="manual", bdm_ref="201.1.2.2.H.7")
    checklist("Horizontal alignment description", status="manual", bdm_ref="201.1.2.2.H.8")
    checklist("Crown or Superelevation", status="manual", bdm_ref="201.1.2.2.H.9")
    checklist("Structure File Number", EXISTING_SFN, "auto", bdm_ref="201.1.2.2.H.10")
    checklist("Date built and year of major rehab", bridge.year_built, "auto", bdm_ref="201.1.2.2.H.11")
    checklist("Disposition of existing bridge after project completion", status="manual", bdm_ref="201.1.2.2.H.12")
else:
    print("No existing bridge loaded. Fill in manually.")

### 4f. Existing Structure Block (201.1.2.2.H)

---

*Located above the Proposed Structure Block, 6.5 in wide on 22x34 sheet*

✅ Brief description of existing bridge type: **Material: 4, Type: 02** *(201.1.2.2.H.1)*

✏️ Span lengths and how measured *(201.1.2.2.H.2)*

...

✏️ Horizontal alignment description *(201.1.2.2.H.8)*

✏️ Crown or Superelevation *(201.1.2.2.H.9)*

✅ Structure File Number: **2102226** *(201.1.2.2.H.10)*

✅ Date built and year of major rehab: **1959** *(201.1.2.2.H.11)*

✏️ Disposition of existing bridge after project completion *(201.1.2.2.H.12)*

In [20]:
section_header("4g. Additional Site Plan Items (201.1.2.2.I-M)")

checklist("Cross section of proposed superstructure (+ elevation of pier types)", status="manual", bdm_ref="201.1.2.2.I")

# Traffic data (auto from TIMS)
if bridge:
    checklist("Design/current ADT", bridge.adt, "auto", bdm_ref="201.1.2.2.J")
    checklist("Future ADT", bridge.future_adt, "auto" if bridge.future_adt else "manual", bdm_ref="201.1.2.2.J")
else:
    checklist("Design and current ADT with clear description", status="manual", bdm_ref="201.1.2.2.J")

# ADTT — check TIMS road inventory for truck percentage
if lat and lon:
    road_inv = ArcGISLayer(TIMS_URLS["road_inventory"])
    road_data = road_inv.query_by_point(lon, lat, buffer_miles=0.25)
    if road_data:
        for rd in road_data[:3]:
            truck_pct = rd.get('TRUCK_PCT') or rd.get('PERCENT_TRUCKS')
            adt_val = rd.get('AADT') or rd.get('ADT')
            if truck_pct and adt_val:
                adtt = int(float(adt_val) * float(truck_pct) / 100)
                checklist("Estimated ADTT (from Road Inventory)", f"{adtt} ({truck_pct}% trucks)", "auto")
                break

checklist("Design ADTT with clear description", status="manual", bdm_ref="201.1.2.2.J")
checklist("Bearing condition for each substructure unit (FIX, EXP, INT)", status="manual", bdm_ref="201.1.2.2.K")
checklist("Navigational clearances", status="manual", bdm_ref="201.1.2.2.L")
checklist("Cross section sketch of proposed abutments", status="manual", bdm_ref="201.1.2.2.M")

### 4g. Additional Site Plan Items (201.1.2.2.I-M)

---

✏️ Cross section of proposed superstructure (+ elevation of pier types) *(201.1.2.2.I)*

✅ Design/current ADT: **41597** *(201.1.2.2.J)*

✅ Future ADT: **57737** *(201.1.2.2.J)*

✏️ Design ADTT with clear description *(201.1.2.2.J)*

✏️ Bearing condition for each substructure unit (FIX, EXP, INT) *(201.1.2.2.K)*

✏️ Navigational clearances *(201.1.2.2.L)*

✏️ Cross section sketch of proposed abutments *(201.1.2.2.M)*

---
## 5. Narrative of Bridge Alternatives (BDM 201.1.2.3)

A brief narrative identifying the structure alternatives and their costs.
This section provides context data to support the narrative.

In [21]:
section_header("5. Narrative of Bridge Alternatives (201.1.2.3)")

checklist("Brief narrative identifying structure alternatives and costs", status="manual", bdm_ref="201.1.2.3")
checklist("Information regarding use of non-standard bridge railing", status="manual", bdm_ref="201.1.2.3")

# Pull nearby bridges for comparison/context
if lat and lon:
    display(Markdown("\n**Nearby bridges for context (within 5 mi):**"))
    nearby = BridgeEngineeringChecks.bridges_near_point(lon, lat, NEARBY_BRIDGE_RADIUS, only_state=True)
    if nearby:
        print(f"Found {len(nearby)} state-maintained bridges within {NEARBY_BRIDGE_RADIUS} mi")
        # Show a summary table
        print(f"{'SFN':<12} {'Facility':<25} {'Feature':<25} {'Year':>5} {'Dist (mi)':>9}")
        print("-" * 80)
        for b in nearby[:15]:
            sfn = b.get('SFN', '?')
            fac = (b.get('STR_LOC_CARRIED') or '?')[:24]
            feat = (b.get('STR_LOC') or '?')[:24]
            yr = b.get('YR_BUILT', '?')
            if isinstance(yr, (int, float)) and yr > 1e10:
                try:
                    from datetime import datetime, timedelta
                    yr = (datetime(1970, 1, 1) + timedelta(milliseconds=yr)).year
                except:
                    yr = '?'
            dist = b.get('distance_miles', '?')
            print(f"{sfn:<12} {fac:<25} {feat:<25} {yr!s:>5} {dist!s:>9}")

### 5. Narrative of Bridge Alternatives (201.1.2.3)

---

✏️ Brief narrative identifying structure alternatives and costs *(201.1.2.3)*

✏️ Information regarding use of non-standard bridge railing *(201.1.2.3)*


**Nearby bridges for context (within 5 mi):**

Found 57 state-maintained bridges within 5.0 mi
SFN          Facility                  Feature                    Year Dist (mi)
--------------------------------------------------------------------------------
2102196      I-71 SB                   3.55 MI N OF FRANKLIN CO  -331516800000      0.02
2102137      I-71 SB                   3.07 N OF FRANKLIN CO     -331516800000      0.47
2102161      I-71 NB                   3.07 N OF FRANKLIN CO     -331516800000      0.48
2102072      I-71 SB                   2.87 MI N OF FRANKLIN CO  -331516800000      0.67
2102102      I-71 NB                   2.87 MI N OF FRANKLIN CO  -331516800000      0.68
2102250      JAYCOX RD.                4.35 MI N OF FRANKLIN CO  -331516800000      0.78
2102021      IR-71 SB                  2.46 N. OF 270             2001      1.08
2102056      IR-71 NB                  2.50 MI N OF IR 270        2001       1.1
2102285      LEWIS CENTER RD           4.98 MI N OF FRANKLIN CO  -331516800000      1.42
21019

In [22]:
# Road classification context
section_header("Road Classification Context")

if lat and lon:
    # Functional classification
    fc_data = TIMSRoad.functional_class_near(lon, lat, radius_miles=0.5)
    if fc_data:
        for fc in fc_data[:3]:
            fc_cd = fc.get('FUNCTION_CLASS_CD') or fc.get('FUNC_CLASS')
            route = fc.get('ROUTE_ID') or fc.get('NLF_ID')
            checklist(f"Functional Class (route {route})", fc_cd, "auto")
    
    # NHS
    nhs_data = TIMSRoad.nhs_routes_near(lon, lat, radius_miles=1.0)
    if nhs_data:
        checklist(f"NHS routes within 1 mi: {len(nhs_data)} found", status="auto")
    else:
        checklist("No NHS routes within 1 mile", status="auto")
    
    # Freight network
    freight_data = TIMSRoad.freight_network_near(lon, lat, radius_miles=2.0)
    if freight_data:
        checklist(f"Freight network routes within 2 mi: {len(freight_data)} found", status="auto")
    else:
        checklist("No freight network routes within 2 miles", status="auto")
else:
    print("No coordinates available.")

### Road Classification Context

---

✅ No NHS routes within 1 mile

✅ No freight network routes within 2 miles

---
## 6. Cost Analysis (BDM 201.1.2.4)

Cost analysis comparing all alternatives — initial construction cost plus major future
rehabilitation and maintenance costs, converted to present dollars.

This section provides historical bid data context and a cost estimation template.

In [23]:
section_header("6. Cost Analysis (201.1.2.4)")

checklist("Cost analysis comparing all alternatives (initial + lifecycle in present dollars)",
          status="manual", bdm_ref="201.1.2.4")

# TODO: Integrate with ODOT Bid Tabs data for unit price lookups.
# The ODOT Price Lookup notebook and Bridge Costs notebook have
# historical bid data that could provide unit prices for:
#   - Concrete (per CY)
#   - Structural steel (per LB)
#   - Prestressed beams (per LF)
#   - Pile driving (per LF)
#   - Drilled shafts (per LF)
#   - Deck area (per SF)
#   - Approach slabs
#   - Removal of existing structure
#
# TODO: Build a bridge cost estimating function that takes:
#   - Bridge type (steel, prestressed, concrete slab)
#   - Span lengths and widths
#   - Foundation type
#   - Location (district for regional cost adjustment)
#   and returns a rough order-of-magnitude cost using
#   historical $/SF deck area from ODOT bid history.
#
# Assumed method: Parametric cost estimation using deck area.
# ODOT tracks average bridge costs per SF of deck area by type.
# A simple model multiplies deck_area * cost_per_sf * location_factor.
# Future maintenance costs are discounted at ODOT's standard discount rate.

display(Markdown("\n**Cost Estimation Template:**"))
print("""
Alternative | Type | Deck Area (SF) | Est. $/SF | Initial Cost | Lifecycle Cost | PV Total
------------|------|----------------|-----------|--------------|----------------|--------
Alt 1       |      |                |           |              |                |
Alt 2       |      |                |           |              |                |
Alt 3       |      |                |           |              |                |
""")

### 6. Cost Analysis (201.1.2.4)

---

✏️ Cost analysis comparing all alternatives (initial + lifecycle in present dollars) *(201.1.2.4)*


**Cost Estimation Template:**


Alternative | Type | Deck Area (SF) | Est. $/SF | Initial Cost | Lifecycle Cost | PV Total
------------|------|----------------|-----------|--------------|----------------|--------
Alt 1       |      |                |           |              |                |
Alt 2       |      |                |           |              |                |
Alt 3       |      |                |           |              |                |



In [24]:
# Nearby project history for cost context
section_header("Project History Near Site (for cost context)")

if lat and lon:
    proj_hist_layer = ArcGISLayer(TIMS_URLS["project_history"])
    hist_projects = proj_hist_layer.query_by_point(lon, lat, buffer_miles=PROJECT_SEARCH_RADIUS)
    if hist_projects:
        print(f"Found {len(hist_projects)} historical projects within {PROJECT_SEARCH_RADIUS} mi:")
        for p in hist_projects[:10]:
            pid = p.get('PID_NBR') or p.get('PID') or '?'
            name = p.get('PROJECT_NME') or p.get('PROJECT_NAME') or '?'
            cost = p.get('TOTAL_COST') or p.get('AWARD_AMT') or '?'
            print(f"  PID {pid}: {name[:50]} (Cost: ${cost})")
    else:
        print("No historical projects found nearby.")

    # Current/upcoming projects
    display(Markdown("\n**Active/Planned Projects (District Work Plan):**"))
    if bridge:
        dwp = TIMSProject.search_near(lon, lat, radius_miles=PROJECT_SEARCH_RADIUS)
    elif lat and lon:
        dwp = TIMSProject.search_near(lon, lat, radius_miles=PROJECT_SEARCH_RADIUS)
    else:
        dwp = []
    
    if dwp:
        print(f"Found {len(dwp)} active/planned projects within {PROJECT_SEARCH_RADIUS} mi:")
        for p in dwp[:10]:
            pid = p.get('PID_NBR') or '?'
            name = p.get('PROJECT_NME') or '?'
            status = p.get('PROJECT_STATUS') or '?'
            print(f"  PID {pid}: {name[:50]} ({status})")
    else:
        print("No active/planned projects found nearby.")
else:
    print("No coordinates available.")

### Project History Near Site (for cost context)

---

No historical projects found nearby.



**Active/Planned Projects (District Work Plan):**

No active/planned projects found nearby.


---
## 7. Foundation Recommendations (BDM 201.1.2.5)

This section gathers geotechnical context. Borehole data from TIMS is queried
automatically; detailed foundation design requires the Geotechnical Report.

In [25]:
section_header("7. Foundation Recommendations (201.1.2.5)")

checklist("General foundation type (bearing/friction piles, drilled shafts, spread footings)",
          status="manual", bdm_ref="201.1.2.5.A")
checklist("Preliminary estimates of nominal and factored resistances",
          status="manual", bdm_ref="201.1.2.5.B")
checklist("All pertinent geotechnical information gathered for study area",
          status="manual", bdm_ref="201.1.2.5")
checklist("Factored bearing pressure/resistance for in-situ material below MSE walls",
          status="manual", bdm_ref="201.1.2.5")
checklist("Retaining wall justification study", status="manual",
          bdm_ref="201.1.2.5 / L&D Vol. 3, 1407.2")

### 7. Foundation Recommendations (201.1.2.5)

---

✏️ General foundation type (bearing/friction piles, drilled shafts, spread footings) *(201.1.2.5.A)*

✏️ Preliminary estimates of nominal and factored resistances *(201.1.2.5.B)*

✏️ All pertinent geotechnical information gathered for study area *(201.1.2.5)*

✏️ Factored bearing pressure/resistance for in-situ material below MSE walls *(201.1.2.5)*

✏️ Retaining wall justification study *(201.1.2.5 / L&D Vol. 3, 1407.2)*

In [26]:
# Query TIMS for nearby boreholes
section_header("Nearby Geotechnical Boreholes (from TIMS)")

if lat and lon:
    geotech_layer = ArcGISLayer(TIMS_URLS["geotech_boreholes"])
    boreholes = geotech_layer.query_by_point(lon, lat, buffer_miles=1.0)
    if boreholes:
        print(f"Found {len(boreholes)} boreholes within 1 mi:")
        for bh in boreholes[:20]:
            bh_id = bh.get('BORE_HOLE_ID') or bh.get('BH_ID') or '?'
            depth = bh.get('TOTAL_DEPTH') or bh.get('DEPTH') or '?'
            pid = bh.get('PID_NBR') or bh.get('PROJECT_ID') or '?'
            print(f"  BH {bh_id} (depth: {depth} ft, PID: {pid})")
    else:
        print("No geotechnical boreholes found within 1 mile.")
        print("A subsurface investigation may be needed.")
else:
    print("No coordinates available.")

### Nearby Geotechnical Boreholes (from TIMS)

---

Found 129 boreholes within 1 mi:
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)
  BH ? (depth: ? ft, PID: ?)


---
## 8. Preliminary Maintenance of Traffic Plan (BDM 201.1.2.6)

The MOT plan must coordinate bridge and roadway construction staging.
This section checks for detour route constraints.

In [27]:
section_header("8. Preliminary Maintenance of Traffic (201.1.2.6)")

checklist("Bridge construction stages match approach roadway stages", status="manual", bdm_ref="201.1.2.6")
checklist("Consistent nomenclature (Stage/Phase 1, 2, etc.)", status="manual", bdm_ref="201.1.2.6")
checklist("Transverse section defining all stages of removal/construction", status="manual", bdm_ref="201.1.2.6")
checklist("Existing superstructure/substructure layout with dimensions (field verified) and photos",
          status="manual", bdm_ref="201.1.2.6.A")
checklist("Type of temporary railing or barrier", status="manual", bdm_ref="201.1.2.6.B")
checklist("Proposed temporary lane widths (clear distance between barriers)",
          status="manual", bdm_ref="201.1.2.6.C")
checklist("Location of cut lines (verify structural integrity)",
          status="manual", bdm_ref="201.1.2.6.D")
checklist("Temporary superelevation modifications for phased construction",
          status="manual", bdm_ref="201.1.2.6.E")
checklist("Width of closure pour (30 in. minimum)", status="manual", bdm_ref="201.1.2.6.F / 309.3.8.5")
checklist("Profile grade, alignment, location/width of temporary structure",
          status="manual", bdm_ref="201.1.2.6.G")
checklist("Location of temporary shoring", status="manual", bdm_ref="201.1.2.6.H")
checklist("Lane closure exceptions with alternative concepts",
          status="manual", bdm_ref="201.1.2.6.I")

### 8. Preliminary Maintenance of Traffic (201.1.2.6)

---

✏️ Bridge construction stages match approach roadway stages *(201.1.2.6)*

✏️ Consistent nomenclature (Stage/Phase 1, 2, etc.) *(201.1.2.6)*

✏️ Transverse section defining all stages of removal/construction *(201.1.2.6)*

...

✏️ Temporary superelevation modifications for phased construction *(201.1.2.6.E)*

✏️ Width of closure pour (30 in. minimum) *(201.1.2.6.F / 309.3.8.5)*

✏️ Profile grade, alignment, location/width of temporary structure *(201.1.2.6.G)*

✏️ Location of temporary shoring *(201.1.2.6.H)*

✏️ Lane closure exceptions with alternative concepts *(201.1.2.6.I)*

In [28]:
# Detour route analysis
section_header("Detour Route Bridge Check")

display(Markdown(
    "Enter SFNs of bridges on the proposed detour route to check for "
    "capacity/condition issues:"
))

# USER INPUT: List the SFNs on the detour route
DETOUR_BRIDGE_SFNS = []  # e.g. ["2103001", "2103002", "2103003"]

if DETOUR_BRIDGE_SFNS and EXISTING_SFN:
    conflicts = BridgeEngineeringChecks.detour_conflict_scan(
        project_bridges=[EXISTING_SFN],
        detour_bridges=DETOUR_BRIDGE_SFNS,
    )
    if conflicts:
        display(Markdown("**Detour route bridge issues found:**"))
        for c in conflicts:
            sfn = c.get('sfn')
            issues = c.get('issues', [c.get('issue', '?')])
            print(f"  SFN {sfn}: {', '.join(issues) if isinstance(issues, list) else issues}")
    else:
        print("No issues found on detour route bridges.")
elif not DETOUR_BRIDGE_SFNS:
    print("No detour bridge SFNs entered. Set DETOUR_BRIDGE_SFNS above to run this check.")

### Detour Route Bridge Check

---

Enter SFNs of bridges on the proposed detour route to check for capacity/condition issues:

No detour bridge SFNs entered. Set DETOUR_BRIDGE_SFNS above to run this check.


---
## 9. Summary & Completeness Check

Overview of all checklist sections with auto-populated vs. manual items.

In [29]:
section_header("Structure Type Study — Summary")

display(Markdown(f"""
| Field | Value |
|-------|-------|
| **SFN** | {EXISTING_SFN or 'N/A (new bridge)'} |
| **PID** | {PID or '(not set)'} |
| **Consultant** | {CONSULTANT or '(not set)'} |
| **Reviewer** | {REVIEWER or '(not set)'} |
| **Coordinates** | {f'{lat}, {lon}' if lat and lon else '(not set)'} |
| **County** | {bridge.county if bridge else '(not set)'} |
| **District** | {bridge.district if bridge else '(not set)'} |
| **Existing Bridge** | {'Yes — data loaded from TIMS' if bridge else 'No existing bridge or not loaded'} |
"""))

if bridge:
    display(Markdown(f"""
**Existing Bridge Quick Facts:**
- Built: {bridge.year_built}
- Type: Material {bridge.main_material}, Design {bridge.main_type}
- Length: {bridge.overall_length} ft ({bridge.total_spans} spans, max {bridge.max_span_length} ft)
- Width: {bridge.deck_width} ft deck, {bridge.roadway_width} ft roadway
- Condition (D/S/S): {bridge.deck_rating}/{bridge.superstructure_rating}/{bridge.substructure_rating}
- Sufficiency: {bridge.sufficiency_rating}
- ADT: {bridge.adt}, Future ADT: {bridge.future_adt}
- Structurally Deficient: {bridge.is_structurally_deficient}
- Scour Critical: {bridge.is_scour_critical}
"""))

display(Markdown("""
**Legend:**
- \u2705 = Auto-populated from TIMS/NHD/USGS
- \u270f\ufe0f = Manual entry required
- \u26a0\ufe0f = Not yet implemented (TODO)
- \u2796 = Not applicable
"""))

### Structure Type Study — Summary

---


| Field | Value |
|-------|-------|
| **SFN** | 2102226 |
| **PID** | (not set) |
| **Consultant** | (not set) |
| **Reviewer** | (not set) |
| **Coordinates** | 40.181756, -82.949253 |
| **County** | DEL |
| **District** | 06 |
| **Existing Bridge** | Yes — data loaded from TIMS |



**Existing Bridge Quick Facts:**
- Built: 1959
- Type: Material 4, Design 02
- Length: 132.0 ft (None spans, max 58.0 ft)
- Width: 67.0 ft deck, 64.0 ft roadway
- Condition (D/S/S): 8/8/8
- Sufficiency: 89.9
- ADT: 41597, Future ADT: 57737
- Structurally Deficient: False
- Scour Critical: False



**Legend:**
- ✅ = Auto-populated from TIMS/NHD/USGS
- ✏️ = Manual entry required
- ⚠️ = Not yet implemented (TODO)
- ➖ = Not applicable


---
## 10. Automation Coverage Assessment

### What This Notebook Automates (~40% of checklist)

| Category | Source | Checklist Sections |
|----------|--------|--------------------|
| Bridge identification and inventory | TIMS ArcGIS REST | 201.1.1, 201.1.2.2.H |
| Condition ratings and load ratings | TIMS | 201.1.1 |
| Traffic data (ADT, ADTT, truck %) | TIMS Road Inventory | 201.1.2.2.J |
| Environmental screening | TIMS (scenic rivers, mussel, wetlands) | 201.1.2.1.D |
| Hydrology context | NHD flowlines, waterbodies, HUC-12 | 201.1.2.1.D |
| Peak flows and basin characteristics | USGS StreamStats | 201.1.2.1.D |
| Stream gauge data | USGS Water Services | 201.1.2.1.D |
| Nearby infrastructure | TIMS (boreholes, rail, bridges) | 201.1.2.5, 201.1.2.2 |
| Road classification (FC, NHS, freight) | TIMS | 201.1.2.3 |
| Project history and DWP | TIMS | 201.1.2.4 |
| Detour route conflict scanning | TIMS + BridgeEngineeringChecks | 201.1.2.6 |

### Gaps Requiring Additional Tools

| Gap | Checklist Impact | Pathway to Close |
|-----|-----------------|------------------|
| **Hydraulic modeling (H and H)** | 201.1.2.1.D.2-D.8 (WSE, backwater, freeboard) | HEC-RAS via ras-commander + rashdf (Section 11) |
| **Alignments and geometry** | 201.1.2.1.A/F/G, 201.1.2.2.B/C/G.8/H.8 | MicroStation in-process extraction (Section 12) |
| **Cost estimation** | 201.1.2.4 | ODOT bid tab integration (TODO) |
| **Drawings / CAD deliverables** | Site plan, cross-sections, profiles | Manual CAD work (cannot automate drawing production) |
| **Foundation design** | 201.1.2.5.A/B | Geotechnical report (PDF parsing possible) |
| **MOT design details** | 201.1.2.6.B-I | Engineering judgment + phasing decisions |
| **Controlled outlets / dams** | 201.1.2.1.E | ODNR dam safety data (no clean API) |

---
## 11. HEC-RAS Hydraulic Modeling Integration

The largest automation gap is hydraulic data: water surface elevations, backwater,
freeboard, velocity, and overtopping -- all required by BDM 201.1.2.1.D.

There is no library called "pyHEC-RAS." The actual Python ecosystem:

| Library | Purpose | Install |
|---------|---------|---------|
| **ras-commander** | Run/modify HEC-RAS projects programmatically | `pip install ras-commander` |
| **rashdf** | Read HEC-RAS HDF5 result files (FEMA-maintained) | `pip install rashdf` |
| **h5py** | Direct HDF5 file access (low-level) | `pip install h5py` |

Since HEC-RAS 5.0+, all unsteady/2D results are stored in `.p##.hdf` files.
The HDF5 hierarchy is documented by USACE and contains water surface elevations,
velocity, shear stress, and geometry data.

In [30]:
# ============================================================
# HEC-RAS Results Extraction Demo
# ============================================================
# pip install rashdf h5py

try:
    import h5py
    H5PY_AVAILABLE = True
except ImportError:
    H5PY_AVAILABLE = False
    print("h5py not installed. Install with: pip install h5py")

# USER INPUT: Path to your HEC-RAS plan HDF file
HECRAS_HDF_PATH = ""  # e.g. r"C:\HEC-RAS\MyProject\MyProject.p01.hdf"

if HECRAS_HDF_PATH and H5PY_AVAILABLE:
    import h5py

    with h5py.File(HECRAS_HDF_PATH, 'r') as hdf:
        print("=== HEC-RAS HDF5 Structure ===")
        print("Top-level groups:")
        for key in hdf.keys():
            print(f"  /{key}")

        if "Geometry" in hdf:
            geom = hdf["Geometry"]
            print("
Geometry sub-groups:")
            for key in geom.keys():
                print(f"  /Geometry/{key}")

            if "Cross Sections" in geom:
                xs = geom["Cross Sections"]
                if "River Station" in xs.attrs:
                    stations = xs.attrs["River Station"]
                    print(f"
Cross Sections: {len(stations)} stations")

        if "Results" in hdf:
            results = hdf["Results"]
            print("
Results sub-groups:")
            def print_tree(group, prefix="  ", depth=0):
                if depth > 3:
                    return
                for key in list(group.keys())[:10]:
                    item = group[key]
                    if isinstance(item, h5py.Group):
                        print(f"{prefix}/{key}/")
                        print_tree(item, prefix + "  ", depth + 1)
                    else:
                        print(f"{prefix}/{key} shape={item.shape} dtype={item.dtype}")
            print_tree(results)

elif not HECRAS_HDF_PATH:
    print("Set HECRAS_HDF_PATH above to explore your HEC-RAS results.")

SyntaxError: unterminated string literal (detected at line 27) (3574428726.py, line 27)

In [31]:
# ============================================================
# Extract STS checklist hydraulic values from HEC-RAS results
# ============================================================

def extract_sts_hydraulics(hdf_path, cross_section_id=None):
    """Extract STS checklist hydraulic values from a HEC-RAS HDF5 file.

    Returns dict with keys matching BDM 201.1.2.1.D checklist items:
        - max_wse_100yr: 100-year water surface elevation (201.1.2.1.D.7)
        - max_velocity_100yr: 100-year velocity (fps)
        - channel_invert: Thalweg/invert elevation at bridge (201.1.2.1.D.5)
    """
    if not H5PY_AVAILABLE:
        return {"error": "h5py not installed"}

    import h5py
    import numpy as np

    results = {}

    with h5py.File(hdf_path, 'r') as hdf:
        # 1D Steady State path
        steady_path = "Results/Steady/Output/Output Blocks/Base Output/Steady Profiles/Cross Sections"
        if steady_path in hdf:
            xs_group = hdf[steady_path]

            if "Water Surface" in xs_group:
                wse = xs_group["Water Surface"][:]
                results["wse_all_profiles"] = wse.tolist()
                results["max_wse_100yr"] = float(np.max(wse[-1, :]))

            if "Velocity Total" in xs_group:
                vel = xs_group["Velocity Total"][:]
                results["max_velocity_100yr"] = float(np.max(vel[-1, :]))

            if "Min Channel Elevation" in xs_group:
                inv = xs_group["Min Channel Elevation"][:]
                results["channel_invert"] = float(np.min(inv))

        # 1D Unsteady path
        unsteady_path = "Results/Unsteady/Output/Output Blocks/Base Output/Unsteady Time Series/Cross Sections"
        if unsteady_path in hdf:
            xs_group = hdf[unsteady_path]
            if "Water Surface" in xs_group:
                wse_ts = xs_group["Water Surface"][:]
                results["max_wse_unsteady"] = float(np.max(wse_ts))

        # 2D Results
        twod_base = "Results/Unsteady/Output/Output Blocks/Base Output/Unsteady Time Series/2D Flow Areas"
        if twod_base in hdf:
            for area_name in hdf[twod_base].keys():
                area = hdf[twod_base][area_name]
                if "Water Surface" in area:
                    wse_2d = area["Water Surface"][:]
                    results[f"max_wse_2d_{area_name}"] = float(np.max(wse_2d))

    return results

# Usage:
if HECRAS_HDF_PATH and H5PY_AVAILABLE:
    hydraulics = extract_sts_hydraulics(HECRAS_HDF_PATH)
    print("=== Extracted Hydraulic Values ===")
    for key, val in hydraulics.items():
        if not key.startswith("wse_all"):
            print(f"  {key}: {val}")

NameError: name 'HECRAS_HDF_PATH' is not defined

In [32]:
# ============================================================
# Using ras-commander to run HEC-RAS from Python
# ============================================================
# pip install ras-commander

try:
    from ras_commander import RasCommander
    RAS_COMMANDER_AVAILABLE = True
except ImportError:
    RAS_COMMANDER_AVAILABLE = False
    print("ras-commander not installed. Install with: pip install ras-commander")

HECRAS_PROJECT_PATH = ""  # e.g. r"C:\HEC-RAS\MyProject\MyProject.prj"

if HECRAS_PROJECT_PATH and RAS_COMMANDER_AVAILABLE:
    rc = RasCommander()
    rc.open_project(HECRAS_PROJECT_PATH)

    plans = rc.get_plan_entries()
    print("Available plans:")
    for plan_id, plan_name in plans.items():
        print(f"  {plan_id}: {plan_name}")

    # Run a plan:
    # plan = rc.get_plan("p01")
    # plan.compute()
    # hydraulics = extract_sts_hydraulics(plan.hdf_path)
elif not HECRAS_PROJECT_PATH:
    print("Set HECRAS_PROJECT_PATH to run HEC-RAS from this notebook.")

ras-commander not installed. Install with: pip install ras-commander
Set HECRAS_PROJECT_PATH to run HEC-RAS from this notebook.


---
## 12. MicroStation / OpenRoads Designer Data Extraction

### The Problem
Bentley's Python API (MSPy* modules) uses **pybind11** (not SWIG) to wrap C++ objects.
The `MSPyMstnPlatform.pyd` links against `USTATION.dll` -- the MicroStation runtime.
These modules **only work inside a running MicroStation process**.

### The Strategy
1. Write exploration/extraction scripts here in this notebook
2. Push the notebook to a machine with MicroStation/ORD installed
3. Run the scripts inside MicroStation (via Python Manager or `pythonrun` keyin)
4. Scripts dump results to JSON files in `~/ms_extraction/`
5. Bring the JSON files back for analysis

### Available APIs (inside MicroStation)
- `MSPyBentley` -- Base types, string utilities
- `MSPyBentleyGeom` -- 3D geometry (points, curves, surfaces, transforms)
- `MSPyECObjects` -- Engineering Classification schema (how ORD stores civil objects)
- `MSPyDgnPlatform` -- DGN file I/O, element iteration, model access
- `MSPyDgnView` -- View manipulation
- `MSPyMstnPlatform` -- Session management, keyin commands, active model access

### Civil-Specific APIs (OpenRoads Designer)
- `MSPyCivilPlatform` -- Alignments, profiles, corridors (if available)
- EC Schema queries -- ORD stores civil objects as EC instances on DGN elements

Each cell below is designed to be **copied and run inside MicroStation**.
Output goes to JSON for offline analysis.

In [33]:
# ============================================================
# STEP 0: Environment Discovery
# ============================================================
# Run FIRST inside MicroStation to discover what is available.
# Execute via: MicroStation > Utilities > Python Manager > Run
# Or keyin: pythonrun path/to/script.py

import json
import sys
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")
    return path

discovery = {
    "python_version": sys.version,
    "python_path": sys.executable,
    "sys_path": sys.path,
    "available_modules": {},
    "civil_modules": {},
}

for mod_name in [
    "MSPyBentley", "MSPyBentleyGeom", "MSPyECObjects",
    "MSPyDgnPlatform", "MSPyDgnView", "MSPyMstnPlatform",
]:
    try:
        mod = __import__(mod_name)
        attrs = [a for a in dir(mod) if not a.startswith("_")]
        discovery["available_modules"][mod_name] = {
            "available": True,
            "attribute_count": len(attrs),
            "sample_attrs": attrs[:50],
        }
    except Exception as e:
        discovery["available_modules"][mod_name] = {
            "available": False, "error": str(e),
        }

for mod_name in [
    "MSPyCivilPlatform", "MSPyCivilGeometry", "MSPyCivilDrainage",
    "MSPyOpenRoads", "MSPySubsurface", "MSPyGeotech",
    "MSPyCivilDesign", "MSPyCorridor",
]:
    try:
        mod = __import__(mod_name)
        attrs = [a for a in dir(mod) if not a.startswith("_")]
        discovery["civil_modules"][mod_name] = {
            "available": True,
            "attribute_count": len(attrs),
            "sample_attrs": attrs[:50],
        }
    except Exception as e:
        discovery["civil_modules"][mod_name] = {
            "available": False, "error": str(e),
        }

save_json("00_environment_discovery.json", discovery)
print(f"Core modules: {sum(1 for v in discovery['available_modules'].values() if v['available'])}/6")
print(f"Civil modules: {sum(1 for v in discovery['civil_modules'].values() if v['available'])}")
for name, info in discovery["civil_modules"].items():
    status = "YES" if info["available"] else f"NO ({info['error'][:60]})"
    print(f"  {name}: {status}")

Saved: C:\Users\dparks1\ms_extraction\00_environment_discovery.json
Core modules: 0/6
Civil modules: 0
  MSPyCivilPlatform: NO (No module named 'MSPyCivilPlatform')
  MSPyCivilGeometry: NO (No module named 'MSPyCivilGeometry')
  MSPyCivilDrainage: NO (No module named 'MSPyCivilDrainage')
  MSPyOpenRoads: NO (No module named 'MSPyOpenRoads')
  MSPySubsurface: NO (No module named 'MSPySubsurface')
  MSPyGeotech: NO (No module named 'MSPyGeotech')
  MSPyCivilDesign: NO (No module named 'MSPyCivilDesign')
  MSPyCorridor: NO (No module named 'MSPyCorridor')


In [ ]:
# ============================================================
# STEP 1: Active File and Model Inventory
# ============================================================
import json
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

from MSPyBentley import *
from MSPyBentleyGeom import *
from MSPyDgnPlatform import *
from MSPyMstnPlatform import *

file_info = {}

try:
    active_file = ISessionMgr.ActiveDgnFile
    file_info["filename"] = str(active_file.GetFileName()) if active_file else None
except Exception as e:
    file_info["filename_error"] = str(e)

try:
    active_model = ISessionMgr.ActiveDgnModelRef
    if active_model:
        file_info["active_model"] = {
            "name": str(active_model.GetModelName()) if hasattr(active_model, "GetModelName") else "unknown",
        }
except Exception as e:
    file_info["active_model_error"] = str(e)

try:
    models = []
    dgn_file = ISessionMgr.ActiveDgnFile
    if dgn_file:
        model_index = dgn_file.GetModelIndex()
        if model_index:
            for entry in model_index:
                models.append({
                    "name": str(entry.GetName()) if hasattr(entry, "GetName") else "?",
                    "description": str(entry.GetDescription()) if hasattr(entry, "GetDescription") else "",
                    "type": str(entry.GetModelType()) if hasattr(entry, "GetModelType") else None,
                })
    file_info["all_models"] = models
except Exception as e:
    file_info["model_enumeration_error"] = str(e)

save_json("01_file_and_models.json", file_info)
print(f"File: {file_info.get('filename', '?')}")
print(f"Models found: {len(file_info.get('all_models', []))}")

In [ ]:
# ============================================================
# STEP 2: Element Scanner -- Catalog What Is in the DGN
# ============================================================
# This is the KEY step -- reveals what ORD objects are stored.

import json
from pathlib import Path
from collections import defaultdict

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

from MSPyBentley import *
from MSPyBentleyGeom import *
from MSPyECObjects import *
from MSPyDgnPlatform import *
from MSPyMstnPlatform import *

model_ref = ISessionMgr.ActiveDgnModelRef
dgn_model = model_ref.GetDgnModel() if model_ref else None

catalog = {
    "by_type": defaultdict(int),
    "by_level": defaultdict(int),
    "by_ec_class": defaultdict(int),
    "total_elements": 0,
    "sample_elements": [],
    "ec_schemas_found": set(),
}

if dgn_model:
    ec_mgr = DgnECManager.GetManager()

    for element in dgn_model.GetGraphicElements():
        catalog["total_elements"] += 1
        elem_type = str(element.GetElementType())
        catalog["by_type"][elem_type] += 1

        try:
            level_id = element.GetLevelId()
            level_cache = dgn_model.GetDgnFile().GetLevelCache()
            level = level_cache.GetLevel(level_id) if level_cache else None
            level_name = str(level.GetName()) if level else str(level_id)
        except Exception:
            level_name = "unknown"
        catalog["by_level"][level_name] += 1

        # EC Class -- how ORD identifies civil objects
        try:
            instances = ec_mgr.FindInstancesOnElement(element, None)
            if instances:
                for inst in instances:
                    class_name = inst.GetClass().GetName()
                    schema_name = inst.GetClass().GetSchema().GetName()
                    full_name = f"{schema_name}:{class_name}"
                    catalog["by_ec_class"][full_name] += 1
                    catalog["ec_schemas_found"].add(schema_name)
        except Exception:
            pass

        if len(catalog["sample_elements"]) < 20:
            sample = {"element_id": element.GetElementId(), "type": elem_type, "level": level_name}
            try:
                rng = element.GetRange()
                if rng:
                    sample["range_low"] = [rng.low.x, rng.low.y, rng.low.z]
                    sample["range_high"] = [rng.high.x, rng.high.y, rng.high.z]
            except Exception:
                pass
            catalog["sample_elements"].append(sample)

catalog["by_type"] = dict(catalog["by_type"])
catalog["by_level"] = dict(catalog["by_level"])
catalog["by_ec_class"] = dict(catalog["by_ec_class"])
catalog["ec_schemas_found"] = list(catalog["ec_schemas_found"])

save_json("02_element_catalog.json", catalog)
print(f"Total elements: {catalog['total_elements']}")
print(f"EC Schemas: {catalog['ec_schemas_found']}")
for cls, c in sorted(catalog["by_ec_class"].items(), key=lambda x: -x[1])[:15]:
    print(f"  {cls}: {c}")

In [ ]:
# ============================================================
# STEP 3: EC Schema Deep Dive
# ============================================================
# Dumps full schema definitions so we can understand what
# properties are available on each civil object type.

import json
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

from MSPyBentley import *
from MSPyECObjects import *
from MSPyDgnPlatform import *
from MSPyMstnPlatform import *

dgn_file = ISessionMgr.ActiveDgnFile
model_ref = ISessionMgr.ActiveDgnModelRef
dgn_model = model_ref.GetDgnModel() if model_ref else None

schema_dump = {"schemas": {}, "total_schemas_found": 0}

try:
    ec_mgr = DgnECManager.GetManager()
    discovered_schemas = set()

    if dgn_model:
        for element in dgn_model.GetGraphicElements():
            try:
                instances = ec_mgr.FindInstancesOnElement(element, None)
                if not instances:
                    continue
                for inst in instances:
                    ec_class = inst.GetClass()
                    schema = ec_class.GetSchema()
                    schema_name = schema.GetName()

                    if schema_name in discovered_schemas:
                        continue
                    discovered_schemas.add(schema_name)

                    schema_info = {
                        "name": schema_name,
                        "classes": {},
                    }
                    try:
                        schema_info["version"] = str(schema.GetVersionString()) if hasattr(schema, "GetVersionString") else "?"
                    except Exception:
                        pass

                    try:
                        for ec_class_entry in schema.GetClasses():
                            class_info = {
                                "name": ec_class_entry.GetName(),
                                "properties": {},
                            }
                            try:
                                for prop in ec_class_entry.GetProperties(True):
                                    class_info["properties"][prop.GetName()] = {
                                        "type": str(prop.GetTypeName()) if hasattr(prop, "GetTypeName") else "?",
                                    }
                            except Exception as pe:
                                class_info["property_error"] = str(pe)

                            schema_info["classes"][ec_class_entry.GetName()] = class_info
                    except Exception as ce:
                        schema_info["class_error"] = str(ce)

                    schema_dump["schemas"][schema_name] = schema_info
            except Exception:
                continue

    schema_dump["total_schemas_found"] = len(discovered_schemas)
except Exception as e:
    schema_dump["error"] = str(e)

save_json("03_ec_schemas.json", schema_dump)
print(f"Schemas found: {schema_dump['total_schemas_found']}")
for name, info in schema_dump["schemas"].items():
    n_classes = len(info.get("classes", {}))
    print(f"  {name}: {n_classes} classes")
    for cls_name in list(info.get("classes", {}).keys())[:5]:
        n_props = len(info["classes"][cls_name].get("properties", {}))
        print(f"    - {cls_name} ({n_props} properties)")

In [ ]:
# ============================================================
# STEP 4: Alignment Extraction
# ============================================================
# Tries multiple approaches to find and extract alignments.

import json
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

from MSPyBentley import *
from MSPyBentleyGeom import *
from MSPyECObjects import *
from MSPyDgnPlatform import *
from MSPyMstnPlatform import *

alignments = {"method_used": None, "alignments": [], "errors": []}

model_ref = ISessionMgr.ActiveDgnModelRef
dgn_model = model_ref.GetDgnModel() if model_ref else None

if dgn_model:
    # Method 1: Try MSPyCivilPlatform directly
    try:
        from MSPyCivilPlatform import *
        alignments["method_used"] = "MSPyCivilPlatform"
        try:
            # API varies by ORD version -- try common patterns
            for getter in ["CivilAlignmentManager", "AlignmentManager", "GeometryManager"]:
                if getter in dir():
                    mgr = eval(f"{getter}.GetManager(dgn_model)")
                    if mgr:
                        for alignment in mgr.GetAlignments():
                            al_data = {"name": str(getattr(alignment, "GetName", lambda: "?")())}
                            try:
                                al_data["length"] = alignment.GetLength()
                                al_data["start_station"] = alignment.GetStartStation()
                            except Exception:
                                pass
                            alignments["alignments"].append(al_data)
                        break
        except Exception as e:
            alignments["errors"].append(f"CivilPlatform API: {e}")
    except ImportError:
        alignments["errors"].append("MSPyCivilPlatform not available")

    # Method 2: Find alignments via EC Class scan
    if not alignments["alignments"]:
        alignments["method_used"] = "EC Class scan"
        try:
            ec_mgr = DgnECManager.GetManager()
            for element in dgn_model.GetGraphicElements():
                try:
                    instances = ec_mgr.FindInstancesOnElement(element, None)
                    if not instances:
                        continue
                    for inst in instances:
                        class_name = inst.GetClass().GetName()
                        if any(kw in class_name.lower()
                               for kw in ["alignment", "linear", "corridor", "profile"]):
                            al_data = {
                                "element_id": element.GetElementId(),
                                "ec_class": f"{inst.GetClass().GetSchema().GetName()}:{class_name}",
                                "properties": {},
                            }
                            try:
                                for prop in inst.GetClass().GetProperties(True):
                                    try:
                                        val = inst.GetValue(prop.GetName())
                                        al_data["properties"][prop.GetName()] = str(val)
                                    except Exception:
                                        pass
                            except Exception:
                                pass
                            alignments["alignments"].append(al_data)
                except Exception:
                    continue
        except Exception as e:
            alignments["errors"].append(f"EC scan: {e}")

    # Method 3: LandXML export suggestion
    if not alignments["alignments"]:
        alignments["method_used"] = "LandXML export (suggested)"
        alignments["suggestion"] = (
            "No alignments found via API. Try exporting LandXML from ORD:\n"
            "  Keyin: CIVILGEOMETRY EXPORT LANDXML\n"
            "  Or: File > Export > LandXML\n"
            "LandXML is standard XML that Python parses with xml.etree.ElementTree."
        )

save_json("04_alignments.json", alignments)
print(f"Method: {alignments['method_used']}")
print(f"Alignments found: {len(alignments['alignments'])}")
for al in alignments["alignments"]:
    print(f"  - {al.get('name') or al.get('ec_class', '?')}")

In [ ]:
# ============================================================
# STEP 5: Terrain / Surface Extraction
# ============================================================

import json
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

from MSPyBentley import *
from MSPyBentleyGeom import *
from MSPyECObjects import *
from MSPyDgnPlatform import *
from MSPyMstnPlatform import *

terrains = {"method_used": None, "surfaces": [], "errors": []}
model_ref = ISessionMgr.ActiveDgnModelRef
dgn_model = model_ref.GetDgnModel() if model_ref else None

if dgn_model:
    # Method 1: Civil terrain API
    try:
        from MSPyCivilPlatform import *
        terrains["method_used"] = "MSPyCivilPlatform"
        # Try common terrain manager patterns
        for getter in ["CivilTerrainManager", "TerrainManager", "SurfaceManager"]:
            if getter in dir():
                try:
                    mgr = eval(f"{getter}.GetManager(dgn_model)")
                    if mgr:
                        for terrain in mgr.GetTerrains():
                            t_data = {"name": str(getattr(terrain, "GetName", lambda: "?")())}
                            terrains["surfaces"].append(t_data)
                        break
                except Exception as e:
                    terrains["errors"].append(f"{getter}: {e}")
    except ImportError:
        terrains["errors"].append("MSPyCivilPlatform not available")

    # Method 2: EC Class scan
    if not terrains["surfaces"]:
        terrains["method_used"] = "EC Class scan"
        ec_mgr = DgnECManager.GetManager()
        for element in dgn_model.GetGraphicElements():
            try:
                instances = ec_mgr.FindInstancesOnElement(element, None)
                if not instances:
                    continue
                for inst in instances:
                    class_name = inst.GetClass().GetName()
                    if any(kw in class_name.lower()
                           for kw in ["terrain", "surface", "dtm", "tin", "mesh"]):
                        t_data = {
                            "element_id": element.GetElementId(),
                            "ec_class": f"{inst.GetClass().GetSchema().GetName()}:{class_name}",
                            "properties": {},
                        }
                        try:
                            for prop in inst.GetClass().GetProperties(True):
                                try:
                                    t_data["properties"][prop.GetName()] = str(inst.GetValue(prop.GetName()))
                                except Exception:
                                    pass
                        except Exception:
                            pass
                        terrains["surfaces"].append(t_data)
            except Exception:
                continue

save_json("05_terrains.json", terrains)
print(f"Method: {terrains['method_used']}")
print(f"Surfaces found: {len(terrains['surfaces'])}")

In [ ]:
# ============================================================
# STEP 6: Substructure / Pier / Abutment Discovery
# ============================================================

import json
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

from MSPyBentley import *
from MSPyBentleyGeom import *
from MSPyECObjects import *
from MSPyDgnPlatform import *
from MSPyMstnPlatform import *

bridge_objects = {
    "piers": [], "abutments": [], "bearings": [],
    "girders": [], "deck": [], "other_bridge": [], "errors": [],
}

BRIDGE_KEYWORDS = [
    "pier", "abutment", "bearing", "girder", "beam", "deck", "slab",
    "diaphragm", "wingwall", "backwall", "cap", "column", "footing",
    "pile", "drilled shaft", "bridge", "superstructure", "substructure",
    "parapet", "railing", "barrier", "approach", "joint",
]

model_ref = ISessionMgr.ActiveDgnModelRef
dgn_model = model_ref.GetDgnModel() if model_ref else None

if dgn_model:
    ec_mgr = DgnECManager.GetManager()
    for element in dgn_model.GetGraphicElements():
        try:
            instances = ec_mgr.FindInstancesOnElement(element, None)
            if not instances:
                continue
            for inst in instances:
                class_name = inst.GetClass().GetName().lower()
                schema_name = inst.GetClass().GetSchema().GetName()
                matched = None
                for kw in BRIDGE_KEYWORDS:
                    if kw in class_name:
                        matched = kw
                        break
                if not matched:
                    continue

                obj_data = {
                    "element_id": element.GetElementId(),
                    "ec_class": f"{schema_name}:{inst.GetClass().GetName()}",
                    "matched_keyword": matched,
                    "properties": {},
                }
                try:
                    for prop in inst.GetClass().GetProperties(True):
                        try:
                            obj_data["properties"][prop.GetName()] = str(inst.GetValue(prop.GetName()))
                        except Exception:
                            pass
                except Exception:
                    pass

                if "pier" in matched or "column" in matched:
                    bridge_objects["piers"].append(obj_data)
                elif "abutment" in matched or "wingwall" in matched or "backwall" in matched:
                    bridge_objects["abutments"].append(obj_data)
                elif "bearing" in matched:
                    bridge_objects["bearings"].append(obj_data)
                elif "girder" in matched or "beam" in matched:
                    bridge_objects["girders"].append(obj_data)
                elif "deck" in matched or "slab" in matched:
                    bridge_objects["deck"].append(obj_data)
                else:
                    bridge_objects["other_bridge"].append(obj_data)
        except Exception:
            continue

save_json("06_bridge_objects.json", bridge_objects)
for cat, items in bridge_objects.items():
    if cat != "errors" and items:
        print(f"  {cat}: {len(items)} objects")

In [ ]:
# ============================================================
# STEP 7: Query TIMS Data INTO MicroStation
# ============================================================
# Pulls TIMS data from inside MicroStation for context.

import json
import requests
from pathlib import Path

OUTPUT_DIR = Path.home() / "ms_extraction"
OUTPUT_DIR.mkdir(exist_ok=True)

def save_json(filename, data):
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"Saved: {path}")

TIMS_BASE = "https://tims.dot.state.oh.us/ags/rest/services"

def query_tims(url, lon=None, lat=None, radius_miles=1.0, where="1=1"):
    params = {"where": where, "outFields": "*", "returnGeometry": "true", "f": "json"}
    if lon and lat:
        deg_lat = radius_miles / 69.0
        deg_lon = radius_miles / 54.6
        params["geometry"] = json.dumps({
            "xmin": lon - deg_lon, "ymin": lat - deg_lat,
            "xmax": lon + deg_lon, "ymax": lat + deg_lat,
            "spatialReference": {"wkid": 4326},
        })
        params["geometryType"] = "esriGeometryEnvelope"
        params["spatialRel"] = "esriSpatialRelIntersects"
        params["inSR"] = "4326"
    try:
        resp = requests.get(f"{url}/query", params=params, timeout=30)
        return [f.get("attributes", {}) for f in resp.json().get("features", [])]
    except Exception as e:
        print(f"TIMS query failed: {e}")
        return []

# USER INPUT
PROJECT_LON = None  # e.g. -82.949
PROJECT_LAT = None  # e.g. 40.182

if PROJECT_LON and PROJECT_LAT:
    bridges = query_tims(f"{TIMS_BASE}/Assets/Bridge_Inventory/MapServer/0",
                         PROJECT_LON, PROJECT_LAT, 2.0)
    boreholes = query_tims(f"{TIMS_BASE}/Assets/Geotech_Bore_Hole_Locations/MapServer/0",
                           PROJECT_LON, PROJECT_LAT, 1.0)
    scenic = query_tims(f"{TIMS_BASE}/Environmental/Scenic_Rivers/MapServer/0",
                        PROJECT_LON, PROJECT_LAT, 2.0)

    context = {
        "project_lon": PROJECT_LON, "project_lat": PROJECT_LAT,
        "nearby_bridges": bridges,
        "nearby_boreholes": boreholes,
        "scenic_rivers": scenic,
    }
    save_json("07_tims_context.json", context)
    print(f"Bridges: {len(bridges)}, Boreholes: {len(boreholes)}, Scenic: {len(scenic)}")
else:
    print("Set PROJECT_LON and PROJECT_LAT to query TIMS.")

In [ ]:
# ============================================================
# STEP 8: PyCharm / VS Code Remote Debug Attachment
# ============================================================
# Develop in PyCharm while running code inside MicroStation.
#
# Setup:
#   1. In PyCharm: Run > Edit Configurations > + > Python Debug Server
#      Set host=localhost, port=5678
#   2. Start the debug server in PyCharm (click Debug)
#   3. Run this script inside MicroStation -- it connects back

# --- Option A: PyCharm pydevd (Professional only) ---
PYCHARM_DEBUG = False  # Set True to enable

if PYCHARM_DEBUG:
    try:
        import pydevd
        pydevd.settrace("localhost", port=5678,
                        stdoutToServer=True, stderrToServer=True, suspend=False)
        print("Connected to PyCharm debugger!")
    except ImportError:
        print("Install: pip install pydevd-pycharm~=<your_version>")
    except ConnectionRefusedError:
        print("Start the debug server in PyCharm first.")

# --- Option B: debugpy (works with VS Code and PyCharm) ---
DEBUGPY_DEBUG = False  # Set True to enable

if DEBUGPY_DEBUG:
    try:
        import debugpy
        debugpy.listen(("localhost", 5678))
        print("debugpy listening on localhost:5678 -- attach your IDE")
        # debugpy.wait_for_client()  # Uncomment to pause until attached
    except ImportError:
        print("Install: pip install debugpy")

print("Set PYCHARM_DEBUG=True or DEBUGPY_DEBUG=True and re-run.")

---
## 13. Bringing Results Back

After running Steps 0-7 inside MicroStation, copy the JSON files from
`~/ms_extraction/` back to your development machine.

### Expected output files:
| File | Contents |
|------|----------|
| `00_environment_discovery.json` | Available Python modules and versions |
| `01_file_and_models.json` | DGN file info and model inventory |
| `02_element_catalog.json` | Element types, levels, EC classes in the DGN |
| `03_ec_schemas.json` | Full EC schema definitions with properties |
| `04_alignments.json` | Extracted alignment geometry and stations |
| `05_terrains.json` | Terrain/surface data with elevation samples |
| `06_bridge_objects.json` | Piers, abutments, girders, deck objects |
| `07_tims_context.json` | TIMS data queried from inside MicroStation |

In [ ]:
# ============================================================
# Load and analyze MicroStation extraction results
# ============================================================
import json
from pathlib import Path

EXTRACTION_DIR = Path.home() / "ms_extraction"
# Or: EXTRACTION_DIR = Path(r"C:\path\to\extracted\json")

def load_extraction(filename):
    path = EXTRACTION_DIR / filename
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    print(f"  Not found: {path}")
    return {}

print(f"Loading from: {EXTRACTION_DIR}\n")

env = load_extraction("00_environment_discovery.json")
if env:
    print("=== Module Availability ===")
    for name, info in env.get("available_modules", {}).items():
        print(f"  {name}: {'OK' if info.get('available') else 'MISSING'}")
    for name, info in env.get("civil_modules", {}).items():
        print(f"  {name}: {'OK' if info.get('available') else 'MISSING'}")

catalog = load_extraction("02_element_catalog.json")
if catalog:
    print(f"\n=== Elements ({catalog.get('total_elements', 0)}) ===")
    for cls, count in sorted(catalog.get("by_ec_class", {}).items(), key=lambda x: -x[1])[:15]:
        print(f"  {cls}: {count}")

schemas = load_extraction("03_ec_schemas.json")
if schemas and schemas.get("schemas"):
    print(f"\n=== EC Schemas ({len(schemas['schemas'])}) ===")
    for name, info in schemas["schemas"].items():
        print(f"  {name} ({len(info.get('classes', {}))} classes)")

alignments = load_extraction("04_alignments.json")
if alignments:
    print(f"\n=== Alignments ({len(alignments.get('alignments', []))}) ===")
    for al in alignments.get("alignments", []):
        print(f"  - {al.get('name') or al.get('ec_class', '?')}")

bridge_obj = load_extraction("06_bridge_objects.json")
if bridge_obj:
    print("\n=== Bridge Objects ===")
    for cat in ["piers", "abutments", "bearings", "girders", "deck", "other_bridge"]:
        items = bridge_obj.get(cat, [])
        if items:
            print(f"  {cat}: {len(items)}")

print("\nShare JSON files for further analysis and API refinement.")